# Notebook 99 — Full-tree smoke test (NOT scientific evidence)

Exercise every module on tiny fixtures. Must finish quickly. Proves the tree is executable.

In [ ]:
# --- notebook preamble: paths, modes, safe display ---
import os, sys, json
from pathlib import Path
REPO = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("FAST_DEV_RUN", "1")
FAST_DEV_RUN = os.environ.get("FAST_DEV_RUN", "1") == "1"
RUN_EXPENSIVE = os.environ.get("RUN_EXPENSIVE", "0") == "1"
RECOMPUTE = os.environ.get("RECOMPUTE", "0") == "1"
try:
    from IPython.display import display
except Exception:
    def display(*a, **k):
        for x in a:
            print(x)
import numpy as np
import subliminal_icl as S
print("FAST_DEV_RUN=", FAST_DEV_RUN, "RUN_EXPENSIVE=", RUN_EXPENSIVE, "pkg", S.__version__)


## 1. Configuration and run manifest

In [ ]:
from subliminal_icl.manifests import Manifest
man = Manifest.create(phase="99", model_tag="smoke", target="eagle", seed=0)
print("run_id:", man.run_id)


## 2. Preflight assertions

In [ ]:
print('smoke: importing all modules')

## 3. Load or compute cached artifacts
Run the fast paths of extract / score / search / cot / stats end to end.

In [ ]:
import subprocess, sys
scripts = ["generate_arithmetic.py --fast", "extract_activations.py --fast",
           "score_candidates.py --fast", "search_contexts.py --fast",
           "clean_replay_eval.py --self-test"]
ok = True
for s in scripts:
    args = s.split()
    r = subprocess.run([sys.executable, str(REPO / "scripts" / args[0]), *args[1:]],
                       capture_output=True, text=True, cwd=str(REPO),
                       env={**os.environ, "FAST_DEV_RUN": "1"})
    print(args[0], "->", "OK" if r.returncode == 0 else "FAIL")
    if r.returncode != 0:
        print(r.stderr[-400:]); ok = False
print("all fast scripts OK:", ok)

## 4. Interactive inspection widgets

## 5. Diagnostic plots and tables

## 6. Scientific gate decision

### ### Interactive inspection (widgets)
In JupyterLab with `ipywidgets`, this section exposes selectors for model / target trait / run id / layer / token position / component / context size / candidate source / search seed / intervention strength, plus row and beam browsers. They are omitted from the executed cells so the notebook also runs headless in `FAST_DEV_RUN`.

In [ ]:
# --- Scientific gate decision + checkpoint ---
from subliminal_icl.gates import run_gate_checks
checks = [("all_fast_scripts_ok", ok, "every fast path executed")]
gs = run_gate_checks("gate_99_smoke", "Full-tree smoke", checks,
                     config={"fast_dev_run": FAST_DEV_RUN}, write=False)
display(gs.to_dataframe())
print("GATE", gs.gate_id, "->", gs.status)
if not FAST_DEV_RUN:
    assert gs.passed, gs.failure_summary


## 7. Write checkpoint report
Smoke run does not write real gate status (FAST_DEV_RUN).